# Billings EDA — Churn Prediction
**Pipeline:** Raw Analysis → Basic Cleaning → EDA (this notebook)

**Target:** Predict customer churn 14 days before subscription renewal date.

---
## 0. Imports & Setup

In [0]:
target = 'target'

In [0]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import math
from scipy.stats import pointbiserialr

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

In [0]:
df_billing = pd.read_csv("../../data/processed/billings.csv")
db_billing_temp = df_billing.copy()

print(f"Shape: {df_billing.shape}")
df_billing.info()

---
## 1. Target Variable Analysis

### 1.1 Unique values & counts

In [0]:
print("Unique values:")
print(df_billing['Prospect_Outcome'].unique())

print("\nValue counts:")
print(df_billing['Prospect_Outcome'].value_counts())

### 1.2 Handle 'Open' records
'Open' means the subscription decision is not yet made — these are not usable for churn prediction. We save them separately and remove from training data.

In [0]:

df_billing_open = df_billing[df_billing['Prospect_Outcome'] == 'Open']
df_billing_open.to_csv('../../data/eda/open_billings.csv', index=False)
df_billing = df_billing[df_billing['Prospect_Outcome'] != 'Open']
print(f"Remaining records: {len(df_billing)}")

### 1.3 Basic distribution

In [0]:
print("Count of each class:")
print(df_billing['Prospect_Outcome'].value_counts())

print("\nPercentage split:")
print(df_billing['Prospect_Outcome'].value_counts(normalize=True).mul(100).round(2))

In [0]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(x='Prospect_Outcome', data=df_billing)

for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom')

plt.title('Distribution of Prospect Outcome')
plt.tight_layout()
plt.show()

### 1.4 Class imbalance check

In [0]:
counts  = df_billing['Prospect_Outcome'].value_counts()
majority = counts.max()
minority = counts.min()
ratio    = majority / minority

print(f"Imbalance Ratio: {ratio:.2f}:1")

if ratio > 3:
    print("Highly imbalanced — will need SMOTE / class_weight handling before modelling")
elif ratio > 1.5:
    print("Moderately imbalanced — consider class_weight adjustment")
else:
    print("Balanced dataset")

### 1.5 Missing values in target

In [0]:
null_count = df_billing['Prospect_Outcome'].isnull().sum()
null_pct   = df_billing['Prospect_Outcome'].isnull().mean() * 100

print(f"Null count : {null_count}")
print(f"Null %     : {null_pct:.2f}%")

df_billing = df_billing.dropna(subset=['Prospect_Outcome'])
print(f"Rows after dropping nulls: {len(df_billing)}")

### 1.6 Encode target
Won = 0 (renewed), Churned = 1 (churned)

In [0]:
df_billing['target'] = df_billing['Prospect_Outcome'].map({'Won': 0, 'Churned': 1})

print(df_billing['target'].value_counts())

---
## 2. Churn Rate Over Time

### 2.1 Churn rate by renewal year

In [0]:
df_billing['Prospect_Renewal_Date'] = pd.to_datetime(df_billing['Prospect_Renewal_Date'])
df_billing['Extracted_Year'] = df_billing['Prospect_Renewal_Date'].dt.year

churn_by_year = df_billing.groupby('Extracted_Year')['target'].mean()
print(churn_by_year)

fig, ax = plt.subplots(figsize=(8, 4))
churn_by_year.plot(kind='bar', ax=ax, alpha=0.7)
ax.plot(range(len(churn_by_year)), churn_by_year.values, marker='o', color='orange', label='trend')
ax.set_title('Churn Rate by Renewal Year')
ax.set_ylabel('Churn Rate')
ax.set_xlabel('Year')
ax.legend()
plt.tight_layout()
plt.show()

# Clean up temporary column
df_billing.drop(columns=['Extracted_Year'], inplace=True)

### 2.2 Churn rate by renewal month

In [0]:
df_billing['Extracted_Month'] = df_billing['Prospect_Renewal_Date'].dt.month

churn_by_month = df_billing.groupby('Extracted_Month')['target'].mean()
print(churn_by_month)

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(churn_by_month.index, churn_by_month.values, alpha=0.7)
ax.plot(churn_by_month.index, churn_by_month.values, marker='o', color='orange', label='trend')
ax.set_title('Churn Rate by Renewal Month')
ax.set_ylabel('Churn Rate')
ax.set_xlabel('Month')
ax.legend()
plt.tight_layout()
plt.show()

# Clean up temporary column
df_billing.drop(columns=['Extracted_Month'], inplace=True)

---
## 3. Segment-Level Churn Analysis

### 3.1 Churn by Band

In [0]:
churn_by_band = pd.crosstab(df_billing['Band'], df_billing['target'], normalize='index')
print(churn_by_band)


churn_by_band[1].sort_values(ascending=False).plot(kind='bar', figsize=(8, 4))
plt.title('Churn Rate by Band')
plt.ylabel('Churn Rate')
plt.xlabel('Band')
plt.tight_layout()
plt.show()

### 3.2 Churn by Tenure Group

In [0]:
churn_by_tenure = pd.crosstab(df_billing['Tenure_Group'], df_billing['target'], normalize='index')
print(churn_by_tenure)

churn_by_tenure[1].sort_values(ascending=False).plot(kind='bar', figsize=(8, 4))
plt.title('Churn Rate by Tenure Group')
plt.ylabel('Churn Rate')
plt.xlabel('Tenure Group')
plt.tight_layout()
plt.show()

### 3.3 Churn by Connection Group

In [0]:
churn_by_connection = pd.crosstab(df_billing['Connection_Group'], df_billing['target'], normalize='index')
print(churn_by_connection)

churn_by_connection[1].sort_values(ascending=False).plot(kind='bar', figsize=(8, 4))
plt.title('Churn Rate by Connection Group')
plt.ylabel('Churn Rate')
plt.xlabel('Connection Group')
plt.tight_layout()
plt.show()

---
## 4. Univariate Analysis — Numerical Columns

### 4.1 Identify numerical columns

In [0]:
numerical_cols = df_billing.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(f"Number of numerical columns: {len(numerical_cols)}")
print(numerical_cols)

### 4.2 Summary statistics

In [0]:
summary = df_billing[numerical_cols].describe().T
summary['skew_flag'] = summary['mean'] - summary['50%']
summary[['count','mean','50%','std','min','max','skew_flag']]

> **Reading skew_flag:**  
Mean ≈ 50% → Normal  
Mean >> 50% → Right skew (tail on the right)  
Mean << 50% → Left skew (tail on the left)

### 4.3 Distribution plots (histogram + KDE)

In [0]:
cols   = numerical_cols
n_cols = 3
n_rows = math.ceil(len(cols) / n_cols)

plt.figure(figsize=(18, 5 * n_rows))

for i, col in enumerate(cols, 1):
    plt.subplot(n_rows, n_cols, i)
    sns.histplot(df_billing[col], kde=True)
    skew = df_billing[col].skew()
    kurt = df_billing[col].kurtosis()
    plt.title(f"{col}\nSkew: {skew:.2f}  Kurt: {kurt:.2f}")

plt.tight_layout()
plt.show()

### 4.4 Skewness classification table

In [0]:
# Skew thresholds:
#  -0.5 to 0.5  → Normal     
#   0.5 to 1.0  → Moderate   
#   > 1.0       → High skew  

dist_summary = []

for col in numerical_cols:
    skew = df_billing[col].skew()
    kurt = df_billing[col].kurtosis()

    if abs(skew) < 0.5:
        skew_type = 'normal'
    elif abs(skew) < 1:
        skew_type = 'moderate'
    else:
        skew_type = 'high'

    dist_summary.append({'column': col, 'skewness': round(skew,2),
                          'kurtosis': round(kurt,2), 'skew_type': skew_type})

dist_summary_df = pd.DataFrame(dist_summary)
dist_summary_df.sort_values('skewness', ascending=False)

### 4.5 Outlier detection — boxplots

In [0]:
n_rows = math.ceil(len(numerical_cols) / 3)

plt.figure(figsize=(18, 5 * n_rows))

for i, col in enumerate(numerical_cols, 1):
    plt.subplot(n_rows, 3, i)
    sns.boxplot(
        y=df_billing[col],
        flierprops=dict(marker='o', markerfacecolor='red',
                        markeredgecolor='red', markersize=5)
    )
    plt.title(col)

plt.tight_layout()
plt.show()

### 4.6 Outlier summary (IQR method)

In [0]:
outlier_summary = []

for col in numerical_cols:
    Q1  = df_billing[col].quantile(0.25)
    Q3  = df_billing[col].quantile(0.75)
    IQR = Q3 - Q1
    lower  = Q1 - 1.5 * IQR
    upper  = Q3 + 1.5 * IQR
    n_out  = ((df_billing[col] < lower) | (df_billing[col] > upper)).sum()

    outlier_summary.append({
        'column': col,
        'outlier_count': n_out,
        'outlier_%': round(n_out / len(df_billing) * 100, 2)
    })

outlier_df = pd.DataFrame(outlier_summary).sort_values('outlier_%', ascending=False)
outlier_df

In [0]:
# ============================================================
# TAKE DECISION
# ------------------------------------------------------------
# Review outlier_df above.
# Decide for each high-outlier column:
#   Option A — Cap at 99th percentile (winsorize)
#   Option B — Log transform (if skew > 1)
#   Option C — Leave as-is (if outliers are genuine business events)
#
# Example (uncomment to apply):
# for col in ['Connection_Net', 'Total_Net_Paid']:
#     upper = df_billing[col].quantile(0.99)
#     df_billing[col] = df_billing[col].clip(upper=upper)
# ============================================================

### 4.7 Zero / constant / negative value check

In [0]:
eda_summary = []

for col in numerical_cols:
    eda_summary.append({
        'column':          col,
        'zero_%':          round((df_billing[col] == 0).mean() * 100, 2),
        'unique_values':   df_billing[col].nunique(),
        'negative_count':  (df_billing[col] < 0).sum()
    })

eda_summary_df = pd.DataFrame(eda_summary)
eda_summary_df

In [0]:
# ============================================================
# TAKE DECISION
# ------------------------------------------------------------
# Review eda_summary_df above.
#
# For high zero_% columns:
#   → Consider creating a binary flag (e.g. Discount_Flag = 1 if Discount_Amount > 0)
#   → Example: df_billing['Discount_Flag'] = (df_billing['Discount_Amount'] > 0).astype(int)
#
# For columns with unique_values == 1 (constant columns):
#   → Drop them — they carry no information
#   → Example: df_billing.drop(columns=['constant_col'], inplace=True)
#
# For columns with unexpected negative values:
#   → Check business logic — are negatives valid (e.g. refunds) or data errors?
# ============================================================

---
## 5. Univariate Analysis — Categorical Columns

### 5.1 Identify categorical columns

In [0]:
cat_cols = df_billing.select_dtypes(include=['object', 'category']).columns.tolist()
print("String/object columns:", cat_cols)

print("\nNumerical columns with < 15 unique values (likely categorical):")
for col in df_billing.select_dtypes(include=['int64','float64']).columns:
    if df_billing[col].nunique() < 15:
        print(f"  {col} → {df_billing[col].nunique()} unique values")

In [0]:
# Add manually identified categorical-style columns
cat_cols += ['Current_Auto_Renewal_Flag', 'target']
cat_cols = list(set(cat_cols))   # remove duplicates

print(f"Final categorical columns ({len(cat_cols)}):")
print(cat_cols)

### 5.2 Value counts & percentage distribution

In [0]:
for col in cat_cols:
    print(f"\n{'='*50}")
    print(f"Column: {col}")
    print("Counts:")
    print(df_billing[col].value_counts(dropna=False))
    print("\nPercentage:")
    print((df_billing[col].value_counts(normalize=True, dropna=False) * 100).round(2))

### 5.3 Summary table

In [0]:
summary = []
for col in cat_cols:
    vc = df_billing[col].value_counts(normalize=True, dropna=False)
    summary.append({
        'column':       col,
        'unique_values': df_billing[col].nunique(),
        'top_category': vc.index[0],
        'top_%':        round(vc.iloc[0] * 100, 2),
        'missing_%':    round(df_billing[col].isna().mean() * 100, 2)
    })

pd.DataFrame(summary).sort_values('top_%', ascending=False)

### 5.4 Identify problem columns

In [0]:
# Low variance — one category dominates > 95%
low_var_cols = [
    col for col in cat_cols
    if df_billing[col].value_counts(normalize=True, dropna=False).iloc[0] > 0.95
]
print("Low variance columns (>95% one value):", low_var_cols)

# High cardinality — too many unique values
high_card_cols = [col for col in cat_cols if df_billing[col].nunique() > 50]
print("High cardinality columns (>50 unique):", high_card_cols)

In [0]:
# ============================================================
# TAKE DECISION
# ------------------------------------------------------------
# Low variance columns → likely safe to drop (no predictive power)
#   → df_billing.drop(columns=low_var_cols, inplace=True)
#
# High cardinality columns → choose one of:
#   Option A — Drop if not useful (e.g. free-text, IDs)
#   Option B — Group rare categories into 'Other'
#   Option C — Target encode (encode by churn rate per category)
# ============================================================

### 5.5 Bar chart visualisations

In [0]:
# Filter to manageable cardinality for plotting
plot_cols = [col for col in cat_cols if df_billing[col].nunique() <= 50]
n_rows    = math.ceil(len(plot_cols) / 3)

plt.figure(figsize=(18, 5 * n_rows))

for i, col in enumerate(plot_cols, 1):
    plt.subplot(n_rows, 3, i)
    df_billing[col].value_counts().head(20).plot(kind='bar', edgecolor='black')
    plt.title(col)
    plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [0]:
num_cols = df_billing.select_dtypes(include=['int64','float64']).columns.tolist()
num_cols = [col for col in num_cols if col != target]

n_rows = math.ceil(len(num_cols) / 3)
plt.figure(figsize=(18, 5 * n_rows))

for i, col in enumerate(num_cols, 1):
    plt.subplot(n_rows, 3, i)
    sns.boxplot(x=target, y=col, data=df_billing)
    plt.title(f"{col} vs Churn")

plt.tight_layout()
plt.show()

---
## 6. Bivariate Analysis — Features vs Churn

In [0]:
target = 'target'

### 6.1 Numerical features vs churn — boxplots

### 6.2 Mean comparison — churned vs renewed

In [0]:
mean_comparison = df_billing.groupby(target)[num_cols].mean().T.round(2)
mean_comparison.columns = ['Renewed (0)', 'Churned (1)']
mean_comparison['diff'] = mean_comparison['Churned (1)'] - mean_comparison['Renewed (0)']
mean_comparison.sort_values('diff', ascending=False)

### 6.3 Categorical features vs churn rate

In [0]:
# Filter out original target string and high cardinality
plot_cat_cols = [
    c for c in cat_cols
    if c not in ['Prospect_Outcome', target] and df_billing[c].nunique() <= 20
]

n_rows = math.ceil(len(plot_cat_cols) / 3)
plt.figure(figsize=(18, 5 * n_rows))

for i, col in enumerate(plot_cat_cols, 1):
    plt.subplot(n_rows, 3, i)
    churn_rate = (
        df_billing.groupby(col, observed=True)[target]
        .mean()
        .sort_values(ascending=False)
        .head(10)
    )
    churn_rate.plot(kind='bar')
    plt.title(f"Churn rate by {col}")
    plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

### 6.4 Score columns comparison — churned vs renewed

In [0]:
score_cols = [
    'Auto_Renewal_Score', 'Total_Renewal_Score_New',
    'Anchoring_Score', 'Sustainability_Score',
    'Status_Scores', 'Tenure_Scores'
]

# Filter to only existing columns
score_cols = [c for c in score_cols if c in df_billing.columns]

df_billing.groupby(target)[score_cols].mean().T.plot(
    kind='bar', figsize=(12, 5), color=['#4C9BE8', '#E85D3C']
)
plt.title('Score comparison: renewed vs churned')
plt.xticks(rotation=45)
plt.legend(['Renewed (0)', 'Churned (1)'])
plt.tight_layout()
plt.show()

### 6.5 Feature engineering — price increase %

In [0]:
df_billing['price_increase_pct'] = (
    (df_billing['Total_Net_Paid'] - df_billing['Last_Years_Price'])
    / df_billing['Last_Years_Price']
) * 100

sns.boxplot(x=target, y='price_increase_pct', data=df_billing)
plt.title('Price increase % vs Churn')
plt.tight_layout()
plt.show()

print(df_billing.groupby(target)['price_increase_pct'].describe())

In [0]:
# ============================================================
# TAKE DECISION
# ------------------------------------------------------------
# Review mean_comparison and the bivariate plots above.
# Note down features that show clear separation between
# churned (1) and renewed (0) groups — those are strong predictors.
#
# Mark features with NO separation as candidates for dropping.
#
# Also decide: keep 'price_increase_pct' as a derived feature?
# ============================================================

---
## 7. Correlation Analysis

### 7.1 Prepare numerical columns for correlation

In [0]:
num_cols = df_billing.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Remove ID columns — they produce meaningless correlations
cols_to_exclude = ['Co_Ref']
num_cols = [col for col in num_cols if col not in cols_to_exclude]

print(f"Numerical columns for correlation: {len(num_cols)}")
print(num_cols)

### 7.2 Feature vs feature correlation heatmap

In [0]:
corr_matrix = df_billing[num_cols].corr()

plt.figure(figsize=(18, 14))
sns.heatmap(
    corr_matrix,
    annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    linewidths=0.5, square=True
)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [0]:
print(corr_matrix)

### 7.3 Extract highly correlated pairs

In [0]:
threshold = 0.8
high_corr_pairs = []

for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        val = corr_matrix.iloc[i, j]
        if abs(val) >= threshold:
            high_corr_pairs.append((
                corr_matrix.columns[i],
                corr_matrix.columns[j],
                round(val, 3)
            ))
            
high_corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)

print(f"{'Feature A':<35} {'Feature B':<35} {'Corr':>6}")
print('-' * 80)
for a, b, v in high_corr_pairs:
    # print(f"{flag} {a:<33} {b:<35} {v:>6}")
    print(f"{a:<33} {b:<35} {v:>6}")

In [0]:
# ============================================================
# TAKE DECISION
# ------------------------------------------------------------
# For each pair (corr >= 0.9): drop one of the two features.
# Rule: keep the one with stronger correlation to 'target'.
#
# For each pair (0.8–0.9): check both against target (next cell),
# then decide which to keep.
#
# Example:
# features_to_drop = ['Amount', 'Starting_Gross']  # fill after review
# df_billing.drop(columns=features_to_drop, inplace=True)
# ============================================================

### 7.4 Feature vs churn — Pearson correlation bar chart

In [0]:
churn_corr = df_billing[num_cols].corr()[target].drop(target)
churn_corr = churn_corr.sort_values(ascending=False)

colors = ['#E24B4A' if v > 0 else '#378ADD' for v in churn_corr.values]

plt.figure(figsize=(10, 10))
churn_corr.plot(kind='barh', color=colors)
plt.axvline(x=0, color='gray', linewidth=0.8)
plt.title('Feature vs Churn — Pearson Correlation')
plt.xlabel('Correlation coefficient')
plt.tight_layout()
plt.show()

print("\n Top features INCREASING churn risk:")
print(churn_corr.head(10))

print("\n Top features REDUCING churn risk:")
print(churn_corr.tail(10))

### 7.5 Point-biserial correlation (statistically correct for binary target)

In [0]:
df_corr = df_billing.copy()

# Replace inf values
df_corr[num_cols] = df_corr[num_cols].replace([np.inf, -np.inf], np.nan)
df_corr = df_corr.dropna(subset=[target])

results = []
for col in num_cols:
    if col == target:
        continue
    temp = df_corr[[col, target]].dropna()
    if len(temp) > 0:
        corr, pval = pointbiserialr(temp[target], temp[col])
        results.append({
            'feature':     col,
            'correlation': round(corr, 4),
            'p_value':     round(pval, 4),
            'significant': 'yes' if pval < 0.05 else 'no'
        })

pb_df = pd.DataFrame(results).sort_values('correlation', ascending=False)
pb_df

### 7.6 Final important features (significant + meaningful correlation)

In [0]:
important_features = pb_df[
    (pb_df['p_value'] < 0.05) & (pb_df['correlation'].abs() > 0.05)
]

print(f"Statistically significant features: {len(important_features)}")
important_features

### 7.7 Categorical features vs churn rate table

In [0]:
cat_cols_obj = df_billing.select_dtypes(include=['object', 'category']).columns
cat_corr = []

for col in cat_cols_obj:
    churn_rate = df_billing.groupby(col)['target'].mean()
    for category, rate in churn_rate.items():
        cat_corr.append({'feature': col, 'category': category, 'churn_rate': round(rate, 3)})

cat_corr_df = pd.DataFrame(cat_corr)
cat_corr_df.sort_values('churn_rate', ascending=False).head(20)

In [0]:
# ============================================================
# TAKE DECISION
# ------------------------------------------------------------
# Use pb_df and cat_corr_df to finalise your feature list:
#
# 1. From pb_df — drop features where:
#    - p_value >= 0.05 (not statistically significant), AND
#    - abs(correlation) < 0.05 (near-zero relationship with churn)
#
# 2. From highly correlated pairs (section 7.3) — drop the weaker one
#    (i.e. the one with lower abs(correlation) in pb_df)
#
# 3. Write final list:
# features_to_drop = []   # fill in after review
# df_billing.drop(columns=features_to_drop, inplace=True)
# ============================================================

---
## 8. Snapshot — Changes Made in This Notebook

In [0]:
added   = df_billing.columns.difference(db_billing_temp.columns).tolist()
removed = db_billing_temp.columns.difference(df_billing.columns).tolist()

print(f"Columns added   ({len(added)}):   {added}")
print(f"Columns removed ({len(removed)}): {removed}")
print(f"\nFinal shape: {df_billing.shape}")

---
## 9. Save EDA Output

In [0]:
# ============================================================
# TAKE DECISION
# ------------------------------------------------------------
# Once all decisions above are made and columns are finalised,
# save the cleaned EDA output for the next stage (feature engineering / modelling).

df_billing.to_csv('/Workspace/Shared/DS_Miniproject/data/eda/billings_eda.csv', index=False)
print(f"Saved: {df_billing.shape}")
# ============================================================

In [0]:
# num_summary = []

# final_num_cols = [
#     # scores
#     'Sustainability_Score',
#     'Total_Renewal_Score_New',
#     'Auto_Renewal_Score',
#     'Anchoring_Score',
#     'Tenure_Scores',
#     'Renewal_Score_At_Release',

#     # tenure & behavior
#     'Tenure_Years',
#     'Payment_Timeframe',
#     '#_of_Connection',
#     'Current_Anchorings',

#     # financial (deduplicated)
#     'Total_Net_Paid',
#     'Last_Years_Price',
#     'Starting_Net',

#     # derived
#     'price_increase_pct',

#     # other
#     'PQQNet',
#     'Starting_PQQ_Net',

   
# ]




# for col in final_num_cols:
#     num_summary.append({
#         'column': col,
#         'dtype': df_billing[col].dtype,
#         'unique_values': df_billing[col].nunique(),
#         'null_%': round(df_billing[col].isna().mean()*100, 2),
#         'sample_values': df_billing[col].dropna().unique()[:5]
#     })

# num_summary_df = pd.DataFrame(num_summary)
# num_summary_df.sort_values(by='unique_values', ascending=False)
# print(num_summary)